# CodeDrift Arena — End-to-End OpenEnv Notebook

Single notebook to install, smoke-test, train, and evaluate a CodeDrift Arena reviewer agent **through the OpenEnv protocol**, with all results streamed to Weights & Biases.

**What this notebook gives you:**
1. W&B login (graphs and run history)
2. Unit tests + a smoke episode (sanity)
3. The OpenEnv server booted and exercised over WebSocket (proves OpenEnv compliance)
4. GRPO training with W&B logging (the main event)
5. Eval: the trained adapter driving the env **via OpenEnv** (not in-process), with rewards plotted and pushed to W&B
6. A self-improvement run using the adaptive-adversary curriculum

Everything you need to run the user's command — `python3 training/train.py --backend hf --difficulty easy --personality random --episodes 500 --steps 100 --output_dir outputs/grpo-qwen15b` — is wired in as the **default** training cell.

**Runtime:** Colab GPU runtime is required (T4 free tier works). The env itself runs on CPU, but training uses 4-bit QLoRA via bitsandbytes which needs CUDA.

## Project overview

### Problem statement

CodeDrift Arena trains a **code-reviewer agent** under **adversarial schema drift**. A frozen "adversary" mutates a synthetic codebase — renames functions, deletes files, changes API signatures — while incoming pull requests still reference the *old* state. The reviewer sees the **current** codebase plus a **stale diff** and must catch every mismatch. A deterministic scorer assigns reward without human labels, which makes the task suitable for GRPO. The challenge: catching subtle drift (a renamed function, a contract change, a deleted module) buried inside otherwise valid-looking code.

*Source: `README.md`, `env/codedrift_env.py`.*

### Environment

`env/codedrift_env.py:69` — `CodeDriftEnv(difficulty, personality, seed)`.

- **`difficulty`** ∈ `{easy, medium, hard}` — caps stale references per episode (1 / 2 / 3).
- **`personality`** ∈ `{random, subtle, aggressive, escalating, adaptive}` — adversary policy:
  - `random` — uniform over drift types
  - `subtle` — biased toward contract changes (hardest to spot)
  - `aggressive` — applies all drift types
  - `escalating` — gets harder every 10 episodes
  - `adaptive` — self-play curriculum, hardens as the reviewer wins (`agents/drift_agent.py:167`)
- **`seed`** — RNG for reproducibility.

The env is **single-step**: `reset()` produces an `Observation` (`prompt`, `pr_diff`, `codebase_context`, `n_stale_refs`); `step(agent_response)` returns `(obs, reward, done=True, info)`.

**Drift catalogue** (`agents/drift_agent.py:29`):
- **rename** — e.g. `getUserData` → `fetchUserData` (8 entries)
- **removal** — e.g. `utils/legacy.py` deleted (6 entries)
- **contract** — e.g. `createOrder(item, qty)` → `createOrder(item, qty, userId)` (5 entries)

### Agent capabilities

The agent receives the rendered prompt and must emit a structured response. The scorer parses three fields (`rewards/scorer.py:244`):

```
VERDICT: APPROVE | REQUEST_CHANGES
ISSUES: <every stale reference found, or 'none'>
REASON: <one-sentence justification>
```

The ground-truth `DriftAction` (`agents/drift_agent.py:84`) carries `drift_type`, `stale_ref`, `current_ref`, and per-type `metadata`. The agent never sees these directly — only their effect in the diff.

### Tasks

A single rollout:
1. `env.reset()` synthesises a codebase, applies drift, renders a stale-aware PR diff.
2. The agent emits `VERDICT / ISSUES / REASON`.
3. `env.step(response)` calls `RewardScorer.score()` and returns the reward + diagnostics.

Tasks-as-code live in `tests/`:
- `test_codedrift_env.py` — episode lifecycle, double-step guard, response truncation
- `test_reward_scorer.py` — scorer correctness, partial credit, false rejection, diff grounding, verbosity penalty
- `test_drift_agent.py` — adversary personality invariants
- `test_server_security.py` — OpenEnv server auth, rate limiting, payload caps

### Reward model / evaluation logic

From `rewards/scorer.py:235`:

| Outcome | Value |
|---|---|
| Caught stale (REQUEST_CHANGES + correct mention) | **+1.0** |
| Correct APPROVE on a clean PR | +0.5 |
| Mentioned stale but said APPROVE (partial) | +0.4 |
| False rejection (REQUEST_CHANGES on clean PR) | −0.3 |
| Spurious mention (cited an off-episode stale ref) | −0.25 each |
| Missed stale ref | **−1.0** |
| Caught-but-ungrounded (stale token absent from diff) | × 0.7 scale |

Plus a verbosity penalty (up to −0.3) when ISSUES is more than 4× the diff length, to discourage catalog-dumping.

Reward is **terminal** (one step per episode) but **dense per signal**: a 3-drift episode can yield up to +3.0 / down to −3.0. The scorer also returns a one-line `metric_strip` for diagnostics, e.g.:

```
reward=+1.00 | recall=100% | verdict=REQUEST_CHANGES | malformed_issues=no | grounded_in_diff=1/1 | spurious=0
```

**Holdout eval** (`training/train.py:50`) is a deterministic split — primary stale-key hashed by seed — so the held-out drifts are stratified by drift type and identifier, not memorised by the model.

### Post-training / self-improvement strategy

The repo's built-in self-improvement is the **adaptive adversary** (`agents/drift_agent.py:167`). When `personality=adaptive`, the env tracks the reviewer's win rate over the last 10 episodes and shifts strategy:
- win-rate > 80% → switch to **aggressive** (more drifts at once)
- 50–80% → **subtle** (contract-heavy, hardest to spot)
- otherwise → **random**

The hackathon-style post-training loop this notebook demonstrates is:
1. Train once on `personality=random` (broad coverage of drift types).
2. Evaluate the resulting adapter through the **OpenEnv** server.
3. Continue training on `personality=adaptive` so the adversary auto-hardens against the now-stronger reviewer.
4. Re-evaluate; compare reward distributions in W&B.

All training runs report to a single W&B project, so curves stack naturally.

## 1. Runtime check

If `nvidia-smi` errors, switch to **Runtime → Change runtime type → GPU (T4)** and re-run.

In [ ]:
!nvidia-smi || echo 'NO GPU — switch the runtime type and re-run.'

## 2. Clone the repo

If your local fixes are not pushed to `bansalbhunesh/codedrift-arena`, push first or replace this cell with a `files.upload()` of a zip.

In [ ]:
import os
BRANCH = 'colab/openenv-end-to-end'
if not os.path.isdir('codedrift-arena'):
    !git clone --branch {BRANCH} https://github.com/bansalbhunesh/codedrift-arena.git
%cd codedrift-arena
!git status -sb

## 3. Install dependencies

Both stacks: server (openenv-core, fastapi, uvicorn, websockets) **and** training (torch, trl, peft, bitsandbytes, transformers, datasets). Plus `wandb` for logging.

We deliberately skip `unsloth` from the install (the optional `--backend unsloth` path) — it's flaky on fresh Colab images and we'll use `--backend hf` here. Add it back with `!pip install unsloth` if you want.

In [ ]:
!pip install -q -r requirements-server.txt
!pip install -q trl transformers peft accelerate bitsandbytes datasets matplotlib numpy wandb

## 4. Weights & Biases login

`training/train.py:330` passes `report_to="wandb"` to `GRPOConfig`, so once you log in here every metric — loss, reward, grad-norm, KL — streams to W&B automatically.

When the next cell runs, paste your API key from <https://wandb.ai/authorize>.

In [ ]:
import os, wandb
wandb.login()  # interactive: paste API key from https://wandb.ai/authorize

os.environ['WANDB_PROJECT'] = 'codedrift-arena'
# Optional: set if you log to a team account
# os.environ['WANDB_ENTITY'] = 'your-team'
os.environ['WANDB_LOG_MODEL'] = 'false'  # don't push base+adapter weights as artifacts (large)
print('W&B project:', os.environ.get('WANDB_PROJECT'),
      '| entity:', os.environ.get('WANDB_ENTITY', '(personal)'))

## 5. Run unit tests (smoke)

All four test files run on CPU in a few seconds and exercise the env, scorer, adversary, and server-security path. A green run here means your repo state is sound before you spend GPU minutes on training.

In [ ]:
!python -m unittest discover -s tests -p 'test_*.py' -v

## 6. Single-episode sanity check (in-process)

Spin up `CodeDriftEnv` directly, send a perfect-oracle response, confirm reward + `metric_strip`. This is the cheapest way to verify the env produces sensible signal.

In [ ]:
import sys
sys.path.insert(0, '.')

from env.codedrift_env import CodeDriftEnv

env = CodeDriftEnv(difficulty='easy', personality='random', seed=7)
obs = env.reset()
print('=== Observation ===')
print('n_stale_refs:', obs.n_stale_refs)
print('prompt (first 600 chars):')
print(obs.prompt[:600])
print('---')

# Cheat: the env's internal action list is the ground truth. We use it ONLY as
# an oracle for this sanity check — the trained agent never sees it.
stale_refs = [a.stale_ref for a in env._actions]
oracle_review = (
    'VERDICT: REQUEST_CHANGES\n'
    f"ISSUES: {', '.join(stale_refs) if stale_refs else 'none'}\n"
    'REASON: stale references detected by oracle.\n'
)
obs2, reward, done, info = env.step(oracle_review)
print('reward:', reward, 'done:', done)
print('metric_strip:', info.get('metric_strip'))
print('episode_outcome:', info.get('episode_outcome'))

## 7. Boot the OpenEnv server

`server/app.py` wraps `CodeDriftEnv` behind `openenv-core` and exposes it over HTTP + WebSocket. The training script does *not* need this — it imports `CodeDriftEnv` in-process — but for the OpenEnv event we **demonstrate compliance** by booting the server and driving an episode through the same surface external agents would use.

In [ ]:
import os, subprocess, time, atexit, urllib.request

os.makedirs('logs', exist_ok=True)
subprocess.run(['pkill', '-f', 'uvicorn server.app:app'], check=False)
time.sleep(1)

log_fh = open('logs/uvicorn.out', 'w')
server_proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_fh, stderr=subprocess.STDOUT,
)
atexit.register(lambda: server_proc.terminate())

ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)
print('server pid:', server_proc.pid, '| ready:', ready)
if not ready:
    print('--- last 60 log lines ---')
    !tail -60 logs/uvicorn.out

In [ ]:
!curl -s http://127.0.0.1:8000/health && echo
!curl -s http://127.0.0.1:8000/metadata | head -c 1500 && echo

## 8. Run an episode through the OpenEnv WebSocket

Stateless `POST /reset` and `POST /step` each spin up a *new* env in `openenv-core`, so a real episode must use the persistent `/ws` connection. The repo's `scripts/openenv_ws_demo.py` does exactly this with a stub review.

In [ ]:
!python scripts/openenv_ws_demo.py --seed 7

## 9. Preview the GRPO dataset

`training/train.py:72` (`build_dataset_rows`) pre-generates episodes offline. Each row carries the prompt, the diff, and the JSON-serialized ground-truth actions used by the reward function. Looking at one row makes the training data obvious.

In [ ]:
from training.train import build_dataset_rows
rows = build_dataset_rows(n_episodes=4, difficulty='easy', personality='random', seed=42)
print(f'rows: {len(rows)} (drifted + clean)')
row = rows[0]
print('keys:', list(row.keys()))
print('n_stale_refs:', row['n_stale_refs'])
print('serialized_actions:', row['serialized_actions'][:300])
print('prompt (first 500 chars):')
print(row['prompt'][:500])

## 10. Train the agent (GRPO)

This is the user-requested command, baked in:

```
python training/train.py --backend hf --difficulty easy --personality random \
    --episodes 500 --steps 100 --output_dir outputs/grpo-qwen15b
```

**What happens:**
- Loads `unsloth/Qwen2.5-1.5B-Instruct` in 4-bit (NF4 + double-quant), wraps with LoRA r=16 on attention projections (`training/train.py:220`).
- Pre-generates 500 episodes (≈ 600 rows including clean PRs) and holds out 20% by stable hash for eval.
- Runs 100 GRPO update steps, 4 generations per prompt, fp16 compute.
- Saves the LoRA adapter to `outputs/grpo-qwen15b/final`.
- Streams loss / reward / KL / grad-norm to W&B every 5 steps (`logging_steps=5`, `report_to="wandb"`).

On a free T4 expect roughly **30–60 minutes** for 100 steps. Tweak `--steps`/`--episodes` down for a quick demo.

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', 'codedrift-arena')
# Name the run so it is easy to find in the W&B UI
os.environ['WANDB_RUN_GROUP'] = 'qwen15b-easy-random'
os.environ['WANDB_NAME'] = 'grpo-easy-random-100steps'

!python training/train.py \
    --backend hf \
    --difficulty easy \
    --personality random \
    --episodes 500 \
    --steps 100 \
    --output_dir outputs/grpo-qwen15b \
    --eval_split_out outputs/grpo-qwen15b/eval_rows.json

## 11. Reward curve from `trainer_state.json`

Local plot for the notebook; the same data lives in W&B with more detail (loss, KL, reward variance, completion length). Click the W&B run URL printed during training to see it.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

state_path = Path('outputs/grpo-qwen15b/trainer_state.json')
if not state_path.exists():
    print('trainer_state.json not found — did training finish?')
else:
    state = json.loads(state_path.read_text())
    log_hist = state.get('log_history', [])
    xs, ys = [], []
    for row in log_hist:
        if 'reward' in row and 'step' in row:
            xs.append(row['step']); ys.append(row['reward'])
    if xs:
        plt.figure(figsize=(8, 4))
        plt.plot(xs, ys, marker='o', linewidth=1)
        plt.title('GRPO reward — easy / random'); plt.xlabel('step'); plt.ylabel('reward')
        plt.grid(alpha=0.3)
        Path('demo').mkdir(exist_ok=True)
        plt.savefig('demo/training_reward_curve.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('saved demo/training_reward_curve.png')
    else:
        print('no reward rows in log_history')

## 12. Eval — drive the trained adapter through OpenEnv

Now we close the loop: load the LoRA adapter on top of the base model, then play held-out episodes **via the OpenEnv WebSocket** — the env scores the agent the same way an external evaluator would.

Each episode:
1. WS `reset` (fresh seed) → observation prompt
2. Local model generates a `VERDICT / ISSUES / REASON` response
3. WS `step` with the response → reward + scorer info

Aggregated rewards are logged to W&B as a separate `eval` run.

In [ ]:
import asyncio, json, os, statistics, torch, websockets, wandb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = 'unsloth/Qwen2.5-1.5B-Instruct'
ADAPTER = 'outputs/grpo-qwen15b/final'
WS_URL = 'ws://127.0.0.1:8000/ws'
N_EPISODES = 16

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(base, ADAPTER)
model.eval()

def review(prompt: str, max_new_tokens: int = 200) -> str:
    msgs = [{'role': 'user', 'content': prompt}]
    inputs = tok.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.95, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, inputs.shape[1]:], skip_special_tokens=True)

async def play_one(seed: int):
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({'type': 'reset', 'data': {'seed': seed}}))
        reset = json.loads(await ws.recv())
        obs = (reset.get('data') or {}).get('observation') or {}
        prompt = obs.get('prompt', '')
        response = review(prompt)
        await ws.send(json.dumps({'type': 'step',
            'data': {'metadata': {'agent_response': response}}}))
        step = json.loads(await ws.recv())
        data = step.get('data') or {}
        info = ((data.get('observation') or {}).get('scorer_info')) or {}
        return data.get('reward', 0.0), info

wandb.init(project=os.environ.get('WANDB_PROJECT', 'codedrift-arena'),
           name='eval-openenv-easy-random', job_type='eval', reinit=True)
rewards = []
for i in range(N_EPISODES):
    r, info = await play_one(seed=1000 + i)
    rewards.append(r)
    wandb.log({'episode': i, 'reward': r,
               'verdict': info.get('verdict'),
               'episode_outcome': info.get('episode_outcome')})
    print(f'ep {i:02d}: r={r:+.2f}  {info.get("metric_strip", "")}')

summary = {'mean': statistics.mean(rewards), 'min': min(rewards),
           'max': max(rewards), 'n': len(rewards)}
wandb.log(summary)
wandb.finish()
print('summary:', summary)

## 13. Self-improvement run — adaptive adversary

Continue training with `--personality adaptive`. The adversary now hardens against the reviewer based on running win-rate (`agents/drift_agent.py:167`). This is the project's built-in curriculum mechanism — no extra training-loop code required.

We point `--output_dir` at a sibling folder so we don't overwrite the first adapter. Both runs land in the same W&B project — switch between them in the W&B UI to compare.

In [ ]:
import os
os.environ['WANDB_RUN_GROUP'] = 'qwen15b-adaptive'
os.environ['WANDB_NAME'] = 'grpo-adaptive-50steps'

# Smaller run by default — bump --steps for a real training pass.
!python training/train.py \
    --backend hf \
    --difficulty medium \
    --personality adaptive \
    --episodes 300 \
    --steps 50 \
    --output_dir outputs/grpo-qwen15b-adaptive

## 14. Save / download the adapter

The LoRA adapter (~50–100 MB depending on rank) lives in `outputs/grpo-qwen15b/final`. Two ways to keep it:

**(a) Download to your machine** — uses Colab's `files.download`. Good for one-shot.

**(b) Push to HuggingFace Hub** — recommended for the hackathon submission. Set `HF_TOKEN` first.

In [ ]:
# (a) Zip + download
import shutil
shutil.make_archive('grpo-qwen15b-adapter', 'zip', 'outputs/grpo-qwen15b/final')
try:
    from google.colab import files
    files.download('grpo-qwen15b-adapter.zip')
except Exception as e:
    print('files.download not available (not on Colab?):', e)

In [ ]:
# (b) Push to HF Hub — uncomment and edit when you are ready
# from huggingface_hub import login, HfApi
# login()  # paste HF token from https://huggingface.co/settings/tokens
# api = HfApi()
# api.create_repo('your-username/codedrift-qwen15b-grpo', exist_ok=True)
# api.upload_folder(
#     folder_path='outputs/grpo-qwen15b/final',
#     repo_id='your-username/codedrift-qwen15b-grpo',
# )

## 15. Shutdown

Stops the OpenEnv server. Optional — Colab releases the runtime on disconnect anyway.

In [ ]:
!pkill -f 'uvicorn server.app:app' || true
print('stopped')